# 06 transfer scvi to fullgene

**Purpose:** Transfer the scVI embedding and cluster labels to the full-gene object.

**Original analysis notebook:** `06_transfer_HVG8000_scVI_clusters_to_fullgene_corrected.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 13 — Transfer HVG8000 scVI clusters to full-gene object

This notebook transfers the selected scVI clustering / UMAP / latent representation from the HVG8000 low-memory branch back onto the full-gene object.

**Main clustering selected for downstream marker DE / annotation:**

- `leiden_scvi_HVG8000_r0_6`

**Refinement / sensitivity clustering kept:**

- `leiden_scvi_HVG8000_r0_7`

The full-gene object remains the biological analysis backbone for marker DE, marker validation, CNV, CopyKAT/SCEVAN, and annotation. The HVG8000 object is used only for scVI integration, UMAP, and clustering.


In [ ]:
# =========================
# 0) Imports and global config
# =========================

import os
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

warnings.filterwarnings("ignore")

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
INTEGRATED_DIR = PROCESSED_DIR / "integrated"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
LOGS_DIR = PROJECT_DIR / "logs"

for d in [INTEGRATED_DIR, METADATA_DIR, FIGURES_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# -------------------------
# Inputs
# -------------------------

FULLGENE_INPUT_H5AD = (
    INTEGRATED_DIR / "GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad"
)

# Prefer the clustering QC h5ad because it contains the resolution sweep columns
# such as leiden_scvi_HVG8000_r0_6 and leiden_scvi_HVG8000_r0_7.
SCVI_CLUSTER_H5AD_CANDIDATES = [
    INTEGRATED_DIR / "GBM_core_4datasets_HVG8000_scVI_clustering_resolution_QC_lowmem.h5ad",
    INTEGRATED_DIR / "GBM_core_4datasets_HVG8000_scVI_UMAP_Leiden_lowmem_Xcounts.h5ad",
    INTEGRATED_DIR / "GBM_core_HVG8000_scVI_UMAP_Leiden.h5ad",
]

# -------------------------
# Selected clustering
# -------------------------

MAIN_CLUSTER_KEY = "leiden_scvi_HVG8000_r0_6"
REFINEMENT_CLUSTER_KEY = "leiden_scvi_HVG8000_r0_7"

# Keep graph matrices only if you really need them in the full-gene object.
# They can be large, so False is safer for 16 GB RAM.
TRANSFER_GRAPH_MATRICES = False

# -------------------------
# Outputs
# -------------------------

FULLGENE_OUTPUT_H5AD = (
    INTEGRATED_DIR / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters_r0_6.h5ad"
)

# Compatibility alias for downstream notebooks
FULLGENE_OUTPUT_ALIAS_H5AD = (
    INTEGRATED_DIR / "GBM_core_4datasets_fullgene_with_HVG8000_scVI_clusters.h5ad"
)

TRANSFER_REPORT_TSV = METADATA_DIR / "transfer_HVG8000_scVI_to_fullgene_report.tsv"
TRANSFERRED_OBS_COLUMNS_TSV = METADATA_DIR / "transfer_HVG8000_scVI_to_fullgene_obs_columns.tsv"
CLUSTER_KEY_SUMMARY_TSV = METADATA_DIR / "fullgene_HVG8000_cluster_key_summary.tsv"
MAIN_CLUSTER_SIZES_TSV = METADATA_DIR / "fullgene_HVG8000_r0_6_cluster_sizes.tsv"
MAIN_CLUSTER_COMPOSITION_TSV = METADATA_DIR / "fullgene_HVG8000_r0_6_cluster_composition.tsv"
TRANSFER_MANIFEST_TSV = METADATA_DIR / "transfer_HVG8000_scVI_to_fullgene_output_manifest.tsv"

SEED = 42

def replace_file(path: Path):
    path = Path(path)
    if path.exists():
        path.unlink()
        print(f"[REPLACED existing file] {path}")

def write_tsv_replace(df: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    replace_file(path)
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")

def write_h5ad_replace(adata_obj, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    replace_file(path)
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")

def choose_first_existing(paths, label):
    print(f"Candidate {label} files:")
    existing = []
    for p in paths:
        print(f"- {p} | exists: {p.exists()}")
        if p.exists():
            existing.append(p)
    if not existing:
        raise FileNotFoundError(f"No {label} file found among candidates.")
    print(f"[USING {label}] {existing[0]}")
    return existing[0]

SCVI_CLUSTER_H5AD = choose_first_existing(SCVI_CLUSTER_H5AD_CANDIDATES, "scVI clustering")
print("\nFull-gene input:", FULLGENE_INPUT_H5AD, "| exists:", FULLGENE_INPUT_H5AD.exists())
print("Full-gene output:", FULLGENE_OUTPUT_H5AD)


In [ ]:
# =========================
# 1) Inspect input objects in backed mode
# =========================

if not FULLGENE_INPUT_H5AD.exists():
    raise FileNotFoundError(f"Full-gene input h5ad not found: {FULLGENE_INPUT_H5AD}")

for label, p in [
    ("fullgene", FULLGENE_INPUT_H5AD),
    ("scvi_cluster", SCVI_CLUSTER_H5AD),
]:
    print("\n" + "=" * 100)
    print(label, p)
    ad = sc.read_h5ad(p, backed="r")
    print(ad)
    print("n_obs:", ad.n_obs)
    print("n_vars:", ad.n_vars)
    print("obs columns preview:", list(ad.obs.columns)[:80])
    print("obsm keys:", list(ad.obsm.keys()))
    print("layers:", list(ad.layers.keys()))
    print("first obs_names:", list(ad.obs_names[:5]))
    print("last obs_names:", list(ad.obs_names[-5:]))
    ad.file.close()


In [ ]:
# =========================
# 2) Read full-gene object and scVI clustering object
# =========================

gc.collect()

adata_full = sc.read_h5ad(FULLGENE_INPUT_H5AD)
adata_scvi = sc.read_h5ad(SCVI_CLUSTER_H5AD)

print("FULL-GENE:", adata_full)
print("scVI/cluster:", adata_scvi)

# Basic integrity checks
if adata_full.n_obs == 0 or adata_scvi.n_obs == 0:
    raise ValueError("One of the input objects has zero cells.")

if adata_full.n_vars < 10000:
    print(
        f"[WARNING] Full-gene object has only {adata_full.n_vars} genes. "
        "Expected ~15k common genes. Please verify the input full-gene object."
    )

print("Full-gene layers:", list(adata_full.layers.keys()))
print("scVI layers:", list(adata_scvi.layers.keys()))

if "counts" not in adata_full.layers:
    print("[WARNING] Full-gene object does not contain layers['counts'].")
else:
    print("[OK] Full-gene object has layers['counts'].")

print("Full-gene X shape:", adata_full.X.shape)
print("scVI X shape:", adata_scvi.X.shape)


In [ ]:
# =========================
# 3) Align cells safely
# =========================

same_order = np.array_equal(adata_full.obs_names.values, adata_scvi.obs_names.values)

print("Same obs_names and same order:", same_order)

if not same_order:
    shared = adata_full.obs_names.intersection(adata_scvi.obs_names)
    print("Shared cells:", len(shared))
    print("Full-gene cells:", adata_full.n_obs)
    print("scVI cells:", adata_scvi.n_obs)

    if len(shared) != adata_full.n_obs:
        missing_in_scvi = adata_full.obs_names.difference(adata_scvi.obs_names)
        missing_in_full = adata_scvi.obs_names.difference(adata_full.obs_names)
        print("Missing in scVI:", len(missing_in_scvi))
        print("Missing in full-gene:", len(missing_in_full))
        raise ValueError(
            "Cell identities do not fully match between full-gene and scVI objects. "
            "Do not transfer until this is resolved."
        )

    print("Reordering scVI object to match full-gene object...")
    adata_scvi = adata_scvi[adata_full.obs_names].copy()

    same_order_after = np.array_equal(adata_full.obs_names.values, adata_scvi.obs_names.values)
    print("Same order after reindex:", same_order_after)
    assert same_order_after

print("[OK] Cell identities and order are aligned.")


In [ ]:
# =========================
# 4) Identify obs columns to transfer
# =========================

# Transfer all Leiden columns from the HVG8000 scVI/QC object.
leiden_cols = [
    c for c in adata_scvi.obs.columns
    if c.startswith("leiden_scvi_HVG8000_") or c.startswith("leiden_scvi_")
]

# Keep important metadata/QC columns if they exist in scVI but not necessarily in full-gene.
candidate_extra_cols = [
    "scvi_batch",
    "core_dataset",
    "dataset_id",
    "geo_id",
    "study_id",
    "sample_id",
    "patient_id",
    "condition",
    "diagnosis",
    "disease_stage",
    "tumor_status",
    "analysis_group_major",
    "analysis_group_detail",
    "ready_for_integration",
]

extra_cols = [c for c in candidate_extra_cols if c in adata_scvi.obs.columns]

obs_cols_to_transfer = []
for c in leiden_cols + extra_cols:
    if c not in obs_cols_to_transfer:
        obs_cols_to_transfer.append(c)

print("Leiden columns found:", leiden_cols)
print("Extra metadata columns to transfer/refresh:", extra_cols)
print("Total obs columns to transfer:", len(obs_cols_to_transfer))

if MAIN_CLUSTER_KEY not in adata_scvi.obs.columns:
    raise KeyError(
        f"Main cluster key {MAIN_CLUSTER_KEY!r} not found in scVI object. "
        f"Available Leiden columns: {leiden_cols}"
    )

if REFINEMENT_CLUSTER_KEY not in adata_scvi.obs.columns:
    print(
        f"[WARNING] Refinement cluster key {REFINEMENT_CLUSTER_KEY!r} not found. "
        "Continuing without it."
    )

# Transfer columns
for col in obs_cols_to_transfer:
    adata_full.obs[col] = adata_scvi.obs[col].copy()

# Standard aliases for downstream notebooks
adata_full.obs["leiden_scvi_main"] = adata_full.obs[MAIN_CLUSTER_KEY].astype(str).astype("category")
adata_full.obs["leiden_scvi_HVG8000_main"] = adata_full.obs[MAIN_CLUSTER_KEY].astype(str).astype("category")

if REFINEMENT_CLUSTER_KEY in adata_full.obs.columns:
    adata_full.obs["leiden_scvi_HVG8000_refinement"] = (
        adata_full.obs[REFINEMENT_CLUSTER_KEY].astype(str).astype("category")
    )

# Save transferred column list
cols_df = pd.DataFrame({
    "obs_column": obs_cols_to_transfer + ["leiden_scvi_main", "leiden_scvi_HVG8000_main", "leiden_scvi_HVG8000_refinement"],
    "present_in_fullgene_after_transfer": [
        c in adata_full.obs.columns for c in obs_cols_to_transfer + ["leiden_scvi_main", "leiden_scvi_HVG8000_main", "leiden_scvi_HVG8000_refinement"]
    ],
})
write_tsv_replace(cols_df, TRANSFERRED_OBS_COLUMNS_TSV)

display(cols_df)


In [ ]:
# =========================
# 5) Transfer scVI latent and UMAP coordinates
# =========================

obsm_keys_to_transfer = []

if "X_scVI" in adata_scvi.obsm.keys():
    obsm_keys_to_transfer.append("X_scVI")
else:
    raise KeyError("X_scVI not found in scVI object; cannot continue.")

if "X_umap" in adata_scvi.obsm.keys():
    obsm_keys_to_transfer.append("X_umap")
else:
    raise KeyError("X_umap not found in scVI object; cannot continue.")

# Optional: transfer other UMAP/embedding keys if present.
for k in ["X_pca"]:
    if k in adata_scvi.obsm.keys():
        obsm_keys_to_transfer.append(k)

for k in obsm_keys_to_transfer:
    adata_full.obsm[k] = adata_scvi.obsm[k].copy()
    print(f"[TRANSFERRED obsm] {k} shape={adata_full.obsm[k].shape}")

print("Transferred obsm keys:", obsm_keys_to_transfer)


In [ ]:
# =========================
# 6) Transfer selected uns and optionally graph matrices
# =========================

# Keep this modest to avoid huge memory/disk bloat.
uns_keys_to_transfer = []

for k in adata_scvi.uns.keys():
    if (
        k.startswith("scVI")
        or k.startswith("umap")
        or k.startswith("neighbors")
        or k in ["hvg", "hvg_info", "clustering_resolution_QC_info", "clustering_resolution_recommendation"]
    ):
        uns_keys_to_transfer.append(k)

for k in uns_keys_to_transfer:
    try:
        adata_full.uns[k] = adata_scvi.uns[k].copy()
        print("[TRANSFERRED uns]", k)
    except Exception as e:
        print("[SKIPPED uns]", k, repr(e))

if TRANSFER_GRAPH_MATRICES:
    for k in adata_scvi.obsp.keys():
        if "connectivities" in k or "distances" in k:
            adata_full.obsp[k] = adata_scvi.obsp[k].copy()
            print("[TRANSFERRED obsp]", k, adata_full.obsp[k].shape)
else:
    print("[INFO] Graph matrices were not transferred because TRANSFER_GRAPH_MATRICES=False.")


In [ ]:
# =========================
# 7) Add transfer metadata and validate final object
# =========================

adata_full.uns["HVG8000_scVI_cluster_transfer_info"] = {
    "description": (
        "Transferred scVI latent representation, UMAP coordinates, and Leiden resolution-sweep "
        "cluster labels from the HVG8000 low-memory scVI branch onto the full-gene object."
    ),
    "source_fullgene_h5ad": str(FULLGENE_INPUT_H5AD),
    "source_scvi_cluster_h5ad": str(SCVI_CLUSTER_H5AD),
    "output_h5ad": str(FULLGENE_OUTPUT_H5AD),
    "main_cluster_key": MAIN_CLUSTER_KEY,
    "main_cluster_aliases": ["leiden_scvi_main", "leiden_scvi_HVG8000_main"],
    "refinement_cluster_key": REFINEMENT_CLUSTER_KEY,
    "n_cells": int(adata_full.n_obs),
    "n_genes": int(adata_full.n_vars),
    "transferred_obs_columns": obs_cols_to_transfer,
    "transferred_obsm_keys": obsm_keys_to_transfer,
    "counts_layer_preserved": bool("counts" in adata_full.layers),
    "important_note": (
        "These are clustering labels, not final biological annotations. "
        "Marker DE/annotation, bias checks, and CNV-based malignant confirmation should follow."
    ),
}

# Validate core fields
required_obs_after = [
    MAIN_CLUSTER_KEY,
    "leiden_scvi_main",
    "leiden_scvi_HVG8000_main",
    "core_dataset",
    "condition",
    "patient_id",
    "sample_id",
]
missing_after = [c for c in required_obs_after if c not in adata_full.obs.columns]
if missing_after:
    raise KeyError(f"Missing required obs columns after transfer: {missing_after}")

required_obsm_after = ["X_scVI", "X_umap"]
missing_obsm_after = [k for k in required_obsm_after if k not in adata_full.obsm.keys()]
if missing_obsm_after:
    raise KeyError(f"Missing required obsm keys after transfer: {missing_obsm_after}")

# Check counts layer has not been lost
has_counts_layer = "counts" in adata_full.layers
print("Final object:", adata_full)
print("has counts layer:", has_counts_layer)
print("n genes:", adata_full.n_vars)
print("X_scVI:", adata_full.obsm["X_scVI"].shape)
print("X_umap:", adata_full.obsm["X_umap"].shape)

for c in required_obs_after:
    print(c, "| present:", c in adata_full.obs.columns, "| NA:", int(adata_full.obs[c].isna().sum()))

if has_counts_layer:
    print("counts layer shape:", adata_full.layers["counts"].shape)


In [ ]:
# =========================
# 8) Reports: cluster sizes and composition for main clustering
# =========================

cluster_key = MAIN_CLUSTER_KEY

cluster_sizes = (
    adata_full.obs[cluster_key]
    .astype(str)
    .value_counts()
    .rename_axis("cluster")
    .reset_index(name="n_cells")
)
cluster_sizes["cluster"] = cluster_sizes["cluster"].astype(str)
cluster_sizes["cluster_key"] = cluster_key
cluster_sizes["cluster_numeric_sort"] = cluster_sizes["cluster"].map(lambda x: int(x) if str(x).isdigit() else 999999)
cluster_sizes = cluster_sizes.sort_values("cluster_numeric_sort").drop(columns=["cluster_numeric_sort"])

write_tsv_replace(cluster_sizes, MAIN_CLUSTER_SIZES_TSV)
display(cluster_sizes)

# Composition by dataset, condition, patient, sample
composition_rows = []

for cluster, sub in adata_full.obs.groupby(cluster_key, observed=True):
    row = {
        "cluster_key": cluster_key,
        "cluster": str(cluster),
        "n_cells": int(sub.shape[0]),
    }

    for col in ["core_dataset", "condition", "patient_id", "sample_id"]:
        if col in sub.columns:
            vc = sub[col].astype(str).value_counts(dropna=False)
            row[f"top_{col}"] = str(vc.index[0]) if len(vc) else "NA"
            row[f"top_{col}_n"] = int(vc.iloc[0]) if len(vc) else 0
            row[f"top_{col}_fraction"] = float(vc.iloc[0] / sub.shape[0]) if len(vc) and sub.shape[0] > 0 else np.nan
            row[f"n_{col}_present"] = int(vc.shape[0])

    composition_rows.append(row)

composition = pd.DataFrame(composition_rows)
composition["cluster_numeric_sort"] = composition["cluster"].map(lambda x: int(x) if str(x).isdigit() else 999999)
composition = composition.sort_values("cluster_numeric_sort").drop(columns=["cluster_numeric_sort"])

write_tsv_replace(composition, MAIN_CLUSTER_COMPOSITION_TSV)
display(composition)


In [ ]:
# =========================
# 9) Cluster key summary across all transferred Leiden resolutions
# =========================

cluster_summary_rows = []

all_cluster_keys = [
    c for c in adata_full.obs.columns
    if c.startswith("leiden_scvi_HVG8000_") or c in ["leiden_scvi_main", "leiden_scvi_HVG8000_main", "leiden_scvi_HVG8000_refinement"]
]

for key in all_cluster_keys:
    counts = adata_full.obs[key].astype(str).value_counts()
    cluster_summary_rows.append({
        "cluster_key": key,
        "n_clusters": int(counts.shape[0]),
        "min_cluster_size": int(counts.min()),
        "max_cluster_size": int(counts.max()),
        "median_cluster_size": float(counts.median()),
        "n_clusters_lt_50_cells": int((counts < 50).sum()),
        "n_clusters_lt_100_cells": int((counts < 100).sum()),
        "n_clusters_lt_500_cells": int((counts < 500).sum()),
        "n_clusters_lt_1000_cells": int((counts < 1000).sum()),
        "is_main": key in [MAIN_CLUSTER_KEY, "leiden_scvi_main", "leiden_scvi_HVG8000_main"],
        "is_refinement": key in [REFINEMENT_CLUSTER_KEY, "leiden_scvi_HVG8000_refinement"],
    })

cluster_key_summary = pd.DataFrame(cluster_summary_rows)
cluster_key_summary = cluster_key_summary.sort_values(["is_main", "cluster_key"], ascending=[False, True])

write_tsv_replace(cluster_key_summary, CLUSTER_KEY_SUMMARY_TSV)
display(cluster_key_summary)


In [ ]:
# =========================
# 10) Save full-gene object with transferred scVI clusters
# =========================

gc.collect()

write_h5ad_replace(adata_full, FULLGENE_OUTPUT_H5AD)

# Compatibility alias for downstream notebooks
write_h5ad_replace(adata_full, FULLGENE_OUTPUT_ALIAS_H5AD)

print("Saved final full-gene object with scVI clusters:")
print(FULLGENE_OUTPUT_H5AD)
print(FULLGENE_OUTPUT_ALIAS_H5AD)


In [ ]:
# =========================
# 11) Final transfer report and manifest
# =========================

report = pd.DataFrame([{
    "source_fullgene_h5ad": str(FULLGENE_INPUT_H5AD),
    "source_scvi_cluster_h5ad": str(SCVI_CLUSTER_H5AD),
    "output_h5ad": str(FULLGENE_OUTPUT_H5AD),
    "output_alias_h5ad": str(FULLGENE_OUTPUT_ALIAS_H5AD),
    "n_cells": int(adata_full.n_obs),
    "n_genes": int(adata_full.n_vars),
    "has_counts_layer": "counts" in adata_full.layers,
    "has_X_scVI": "X_scVI" in adata_full.obsm,
    "X_scVI_shape": str(adata_full.obsm["X_scVI"].shape) if "X_scVI" in adata_full.obsm else "NA",
    "has_X_umap": "X_umap" in adata_full.obsm,
    "X_umap_shape": str(adata_full.obsm["X_umap"].shape) if "X_umap" in adata_full.obsm else "NA",
    "main_cluster_key": MAIN_CLUSTER_KEY,
    "main_cluster_key_present": MAIN_CLUSTER_KEY in adata_full.obs.columns,
    "main_cluster_alias_present": "leiden_scvi_main" in adata_full.obs.columns,
    "main_n_clusters": int(adata_full.obs[MAIN_CLUSTER_KEY].astype(str).nunique()),
    "refinement_cluster_key": REFINEMENT_CLUSTER_KEY,
    "refinement_cluster_key_present": REFINEMENT_CLUSTER_KEY in adata_full.obs.columns,
    "n_transferred_obs_columns": len(obs_cols_to_transfer),
    "transferred_obsm_keys": ",".join(obsm_keys_to_transfer),
    "graph_matrices_transferred": TRANSFER_GRAPH_MATRICES,
}])

write_tsv_replace(report, TRANSFER_REPORT_TSV)
display(report)

manifest = pd.DataFrame([
    {"name": "source_fullgene_h5ad", "path": str(FULLGENE_INPUT_H5AD), "exists": FULLGENE_INPUT_H5AD.exists()},
    {"name": "source_scvi_cluster_h5ad", "path": str(SCVI_CLUSTER_H5AD), "exists": SCVI_CLUSTER_H5AD.exists()},
    {"name": "output_fullgene_h5ad", "path": str(FULLGENE_OUTPUT_H5AD), "exists": FULLGENE_OUTPUT_H5AD.exists()},
    {"name": "output_fullgene_alias_h5ad", "path": str(FULLGENE_OUTPUT_ALIAS_H5AD), "exists": FULLGENE_OUTPUT_ALIAS_H5AD.exists()},
    {"name": "transfer_report_tsv", "path": str(TRANSFER_REPORT_TSV), "exists": TRANSFER_REPORT_TSV.exists()},
    {"name": "transferred_obs_columns_tsv", "path": str(TRANSFERRED_OBS_COLUMNS_TSV), "exists": TRANSFERRED_OBS_COLUMNS_TSV.exists()},
    {"name": "cluster_key_summary_tsv", "path": str(CLUSTER_KEY_SUMMARY_TSV), "exists": CLUSTER_KEY_SUMMARY_TSV.exists()},
    {"name": "main_cluster_sizes_tsv", "path": str(MAIN_CLUSTER_SIZES_TSV), "exists": MAIN_CLUSTER_SIZES_TSV.exists()},
    {"name": "main_cluster_composition_tsv", "path": str(MAIN_CLUSTER_COMPOSITION_TSV), "exists": MAIN_CLUSTER_COMPOSITION_TSV.exists()},
])

write_tsv_replace(manifest, TRANSFER_MANIFEST_TSV)
display(manifest)


In [ ]:
# =========================
# 12) Quick UMAP sanity plots
# =========================

# These plots are only for visual sanity checking.
# The main marker-based annotation will happen in the next notebooks.

plot_cols = [
    "leiden_scvi_main",
    "core_dataset",
    "condition",
    "patient_id",
]

plot_cols = [c for c in plot_cols if c in adata_full.obs.columns]

sc.pl.umap(
    adata_full,
    color=plot_cols,
    ncols=2,
    wspace=0.5,
    frameon=False,
)
